# Chapter 2 — Function Calling and Structured Outputs · Practice

**Goal:** wire the Responses API function-calling loop by hand, enforce strict schemas, steer calls with `tool_choice`, get typed data back with Structured Outputs, and finish with a small customer-support agent.

- Theory: [README.md](README.md) (sections 2.1–2.8). This notebook follows the same numbering.
- **Estimated time:** 2–3 hours
- **Estimated API cost:** < $0.01 with the default `gpt-6-luna` (only the cells marked **(live)** call the API; everything else runs offline on fake `Response` objects built from the real SDK types)
- **Prerequisites:** [00-prerequisites](../../00-prerequisites/README.md) (venv + `.env`), [Chapter 1](../01-responses-api-fundamentals/README.md)

## Setup

In [ ]:
import json
import os
from typing import Literal, Optional

import openai
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv(find_dotenv())
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")   # cheap default; try gpt-6.1-sol for harder multi-step tool use
LIVE_KW = {"reasoning": {"effort": "low"}, "max_output_tokens": 1500}   # keep live calls small and cheap
print("openai SDK", openai.__version__, "| model:", MODEL)

The next cell needs a real key. If you don't have one yet, skip it: every cell not marked **(live)** still works.

In [ ]:
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env (see 00-prerequisites)"
client = OpenAI()

Offline helpers. `fake_response()` builds a **real** `openai.types.responses.Response` from dicts, so the offline code below reads exactly the same attributes as live code (`response.output`, `item.call_id`, `response.output_text`, ...). `ScriptedClient` replays a list of fake responses and records every request, so you can run a whole tool loop without an API key.

In [ ]:
from openai.types.responses import Response


def fake_response(*items: dict, status: str = "completed", incomplete_reason: str | None = None,
                  rid: str = "resp_fake") -> Response:
    """Build a real SDK `Response` object from plain dicts, so offline code sees the same types as live code."""
    return Response.model_validate({
        "id": rid, "object": "response", "created_at": 0, "model": "offline-fake",
        "status": status,
        "incomplete_details": {"reason": incomplete_reason} if incomplete_reason else None,
        "parallel_tool_calls": True, "tool_choice": "auto", "tools": [],
        "output": list(items),
    })


def fc(name: str, args: dict, call_id: str) -> dict:
    """A `function_call` output item. Note: `arguments` is a JSON *string*."""
    return {"type": "function_call", "id": "fc_" + call_id.split("_")[-1], "call_id": call_id,
            "name": name, "arguments": json.dumps(args), "status": "completed"}


def msg(text: str, phase: str | None = None) -> dict:
    """An assistant `message` output item with one `output_text` part (optionally with a `phase`)."""
    item = {"type": "message", "id": "msg_fake", "role": "assistant", "status": "completed",
            "content": [{"type": "output_text", "text": text, "annotations": []}]}
    if phase is not None:
        item["phase"] = phase          # "commentary" (preamble) or "final_answer"
    return item


def refusal(text: str) -> dict:
    """An assistant `message` whose content is a `refusal` part instead of `output_text`."""
    return {"type": "message", "id": "msg_refusal", "role": "assistant", "status": "completed",
            "content": [{"type": "refusal", "refusal": text}]}


def reasoning(rid: str = "rs_fake") -> dict:
    """A (summary-less) reasoning item, as reasoning models return alongside tool calls."""
    return {"type": "reasoning", "id": rid, "summary": []}


class ScriptedClient:
    """Stands in for `OpenAI()`: `client.responses.create(**kw)` returns pre-scripted responses
    in order and records a snapshot of every request, so loops can run (and be checked) offline."""

    def __init__(self, scripted: list[Response]):
        self._queue = list(scripted)
        self.requests: list[dict] = []
        self.responses = self          # so `client.responses.create(...)` works

    def create(self, **kwargs) -> Response:
        snapshot = dict(kwargs)
        if isinstance(snapshot.get("input"), list):
            snapshot["input"] = list(snapshot["input"])   # copy: the caller keeps mutating its list
        self.requests.append(snapshot)
        if not self._queue:
            raise RuntimeError("ScriptedClient: no scripted responses left (loop did not stop?)")
        return self._queue.pop(0)


def item_type(x) -> str | None:
    """Type of an input/output item, whether it is an SDK object or a plain dict."""
    return x.get("type") if isinstance(x, dict) else getattr(x, "type", None)

In [ ]:
# Fake in-memory "backend" for the customer-support examples. No real systems are touched.
CUSTOMERS = {
    "C-001": {"customer_id": "C-001", "name": "Jane Doe", "email": "jane@example.com",
              "status": "active", "order_ids": ["A-1001", "A-1002"]},
    "C-002": {"customer_id": "C-002", "name": "Sam Lee", "email": "sam@example.com",
              "status": "active", "order_ids": ["A-2001"]},
}
ORDERS = {
    "A-1001": {"order_id": "A-1001", "customer_id": "C-001", "status": "delivered",
               "total_usd": 89.90, "items": ["Wireless mouse", "USB-C cable"]},
    "A-1002": {"order_id": "A-1002", "customer_id": "C-001", "status": "shipped",
               "total_usd": 249.00, "items": ["Mechanical keyboard"]},
    "A-2001": {"order_id": "A-2001", "customer_id": "C-002", "status": "delivered",
               "total_usd": 35.50, "items": ["Phone case"]},
}


get_customer_tool = {
    "type": "function",
    "name": "get_customer",
    "description": (
        "Look up ONE customer by email or by customer ID and return name, account status and "
        "their order IDs (not order details). Call this FIRST to verify who you are talking to, "
        "before any refund. Pass exactly one of email / customer_id and set the other to null."
    ),
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": ["string", "null"], "description": "Customer email, e.g. jane@example.com, or null"},
            "customer_id": {"type": ["string", "null"], "description": "Customer ID like C-001, or null"},
        },
        "required": ["email", "customer_id"],
        "additionalProperties": False,
    },
}

lookup_order_tool = {
    "type": "function",
    "name": "lookup_order",
    "description": (
        "Look up ONE order by order ID and return its status, items, total in USD and owning "
        "customer ID. Use it for any question about an order's status or contents. Do NOT use it "
        "to identify a customer. Order IDs look like A-1001."
    ),
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {"order_id": {"type": "string", "description": "Order ID, e.g. A-1001"}},
        "required": ["order_id"],
        "additionalProperties": False,
    },
}

process_refund_tool = {
    "type": "function",
    "name": "process_refund",
    "description": (
        "Refund all or part of a DELIVERED order that belongs to the verified customer. "
        "Irreversible. Only call after get_customer and lookup_order, and only for the amount "
        "the customer asked for (never more than the amount still refundable on the order). Refunds above 100 USD are "
        "not executed; they return requires_human_approval."
    ),
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {
            "order_id": {"type": "string", "description": "Order ID, e.g. A-1001"},
            "amount_usd": {"type": "number", "minimum": 0, "description": "Refund amount in USD"},
            "reason": {"type": "string", "enum": ["damaged", "not_as_described", "late", "other"]},
        },
        "required": ["order_id", "amount_usd", "reason"],
        "additionalProperties": False,
    },
}

SUPPORT_TOOLS = [get_customer_tool, lookup_order_tool, process_refund_tool]


print("customers:", list(CUSTOMERS), "| orders:", list(ORDERS))
print("tools:", [t["name"] for t in SUPPORT_TOOLS])

## 2.1 The function-calling round trip

The model returns a `function_call` **output item** with a `call_id` and JSON-encoded `arguments`. Your code runs the function and sends back a `function_call_output` **input item** with the same `call_id`. Reasoning items that come back with the call must be replayed too. [Theory](README.md#21-the-function-calling-round-trip)

In [ ]:
# What a reasoning model sends back when it wants a tool (offline fake, real SDK types)
first = fake_response(
    reasoning(),
    fc("lookup_order", {"order_id": "A-1001"}, call_id="call_abc123"),
)
for item in first.output:
    print(type(item).__name__, "| type =", item.type)

call = first.output[1]
print("\nitem id   :", call.id)          # fc_...   identifies the ITEM
print("call_id   :", call.call_id)       # call_... links the OUTPUT to this call
print("arguments :", repr(call.arguments), "->", type(call.arguments).__name__)
args = json.loads(call.arguments)        # a JSON string, so decode it
print("decoded   :", args)

In [ ]:
# Run the "function" and build the next request's input by hand
input_list: list = [{"role": "user", "content": "Where is order A-1001?"}]
input_list += first.output                              # replay EVERY item, reasoning included

result = ORDERS[args["order_id"]]
input_list.append({
    "type": "function_call_output",
    "call_id": call.call_id,                            # not call.id
    "output": json.dumps(result),                       # a string; JSON is a good default
})
for x in input_list:
    print(item_type(x) or "message(user)")

> **Observe:** the second request carries four items: the user message, the reasoning item, the `function_call`, and your `function_call_output`. Nothing is wrapped in a "tool message" as in Chat Completions.

### Function tools vs built-in tools: who executes the call?

One request can mix your functions with **hosted built-in tools** such as `web_search` and `file_search`. Hosted calls come back already executed; only `function_call` items are yours to run. Use a built-in tool when one exists, and write a function for capabilities that are yours. [Theory](README.md#function-tools-vs-built-in-tools)

In [ ]:
# A response from a request with tools=[{"type": "file_search", ...}, {"type": "web_search"}, lookup_order_tool]
mixed = fake_response(
    {"type": "file_search_call", "id": "fs_1", "status": "completed",
     "queries": ["refund policy for damaged items"], "results": None},
    {"type": "web_search_call", "id": "ws_1", "status": "completed",
     "action": {"type": "search", "query": "carrier delays this week"}},
    fc("lookup_order", {"order_id": "A-1001"}, call_id="call_mix1"),
)

HOSTED_CALL_TYPES = {"web_search_call", "file_search_call", "image_generation_call", "mcp_call"}

def calls_to_execute(response) -> list:
    """Only function calls are ours. Hosted call items already ran at OpenAI: log them, don't run them."""
    mine = []
    for item in response.output:
        if item.type == "function_call":
            mine.append(item)
        elif item.type in HOSTED_CALL_TYPES:
            print(f"hosted, already executed by OpenAI: {item.type} ({item.id})")
    return mine

todo = calls_to_execute(mixed)
print("ours to run:", [(c.name, c.call_id) for c in todo])
assert [c.call_id for c in todo] == ["call_mix1"]

> **Observe:** three tool calls came back, but only one `function_call_output` is needed in the next request. Client-executed built-in tools (`computer`, local `shell`, `apply_patch`) are different again: you run them and return their own `*_output` items. Chapter 4 covers them.

> **Try it:** the policy PDFs are uploaded to a vector store. Would you write a `search_policies()` function or use `file_search`? Write down one requirement that would change your answer (for example, a data-residency rule or a custom ranking).

Now the function round trip against the real API **(live)**:

In [ ]:
# (live) one tool, one round trip
live_input: list = [{"role": "user", "content": "What's the status of order A-1002?"}]
resp = client.responses.create(model=MODEL, tools=[lookup_order_tool], input=live_input, **LIVE_KW)
live_input += resp.output
for item in resp.output:
    if item.type == "function_call":
        print("model asked for:", item.name, item.arguments)
        live_input.append({"type": "function_call_output", "call_id": item.call_id,
                           "output": json.dumps(ORDERS.get(json.loads(item.arguments)["order_id"]))})

resp = client.responses.create(model=MODEL, tools=[lookup_order_tool], input=live_input, **LIVE_KW)
print(resp.status, "|", resp.output_text)
print("usage:", resp.usage.input_tokens, "in /", resp.usage.output_tokens, "out")

> **Try it:** print `[i.type for i in resp.output]` after the first call. Then ask about an order that doesn't exist (`A-9999`) and see what the model does with an output of `null`.

## 2.2 Defining function tools: Responses vs Chat Completions shape

Responses tools are **flat** (`type`, `name`, `description`, `parameters`, `strict`). Chat Completions tools are **nested** under a `function` key. Descriptions are the model's main signal for choosing a tool, so write them like documentation for a new colleague. [Theory](README.md#22-defining-function-tools-responses-vs-chat-completions-shape)

In [ ]:
def to_chat_tool(tool: dict) -> dict:
    """Responses (flat) -> Chat Completions (nested). Useful when you must support both APIs."""
    return {"type": "function", "function": {k: v for k, v in tool.items() if k != "type"}}

print("Responses shape keys      :", list(lookup_order_tool))
chat_shape = to_chat_tool(lookup_order_tool)
print("Chat Completions shape keys:", list(chat_shape), "->", list(chat_shape["function"]))

In [ ]:
# The SDK's Pydantic helper produces the *Chat Completions* shape
class LookupOrder(BaseModel):
    """Look up one order by order ID and return status, items and total."""
    order_id: str

helper_tool = openai.pydantic_function_tool(LookupOrder, name="lookup_order")
print(json.dumps(helper_tool, indent=2))

> **Observe:** the helper's output is nested under `"function"`. `client.responses.parse(tools=[helper_tool])` converts it for you (and fills `call.parsed_arguments`), but `client.responses.create(tools=[...])` expects the flat shape. Writing the converter is homework exercise 1.

> **Try it:** shorten `lookup_order_tool["description"]` to "Gets an order." and ask yourself the "intern test" from the README: could a new colleague tell when to use it instead of `get_customer`?

## 2.3 Strict mode schema rules

With `strict: true`, every object needs `additionalProperties: false` and **every** property must be in `required`; optional values are expressed as `["type", "null"]`. If you omit `strict` in Responses, the API tries strict mode and silently falls back if your schema doesn't qualify. [Theory](README.md#23-strict-mode-schema-rules)

In [ ]:
def strict_problems(schema: dict, path: str = "$") -> list[str]:
    """Report violations of OpenAI's strict-mode rules (offline lint, not a full validator)."""
    problems = []
    types = schema.get("type")
    is_object = types == "object" or (isinstance(types, list) and "object" in types)
    if is_object:
        props = schema.get("properties", {})
        if schema.get("additionalProperties") is not False:
            problems.append(f"{path}: additionalProperties must be false")
        missing = set(props) - set(schema.get("required", []))
        if missing:
            problems.append(f"{path}: not in required: {sorted(missing)} (make them nullable instead)")
        for name, sub in props.items():
            problems += strict_problems(sub, f"{path}.{name}")
    if isinstance(schema.get("items"), dict):
        problems += strict_problems(schema["items"], f"{path}[]")
    for i, sub in enumerate(schema.get("anyOf", [])):
        problems += strict_problems(sub, f"{path}.anyOf[{i}]")
    for name, sub in schema.get("$defs", {}).items():
        problems += strict_problems(sub, f"$defs.{name}")
    return problems

loose = {
    "type": "object",
    "properties": {
        "email": {"type": "string"},
        "customer_id": {"type": "string"},
        "address": {"type": "object", "properties": {"city": {"type": "string"}}},
    },
    "required": ["email"],
}
print("loose schema   :", *strict_problems(loose), sep="\n  ")
for tool in SUPPORT_TOOLS:
    print(f"{tool['name']:15}:", strict_problems(tool["parameters"]) or "OK")

In [ ]:
# What Pydantic + the SDK send for a model with Optional fields and a default
class RefundRequest(BaseModel):
    order_id: str
    amount_usd: float
    note: Optional[str] = None          # Python default... but still required in the strict schema

schema = openai.pydantic_function_tool(RefundRequest)["function"]["parameters"]
print("required:", schema["required"])
print("note    :", schema["properties"]["note"])
print("additionalProperties:", schema["additionalProperties"])
print("lint    :", strict_problems(schema) or "OK")

> **Observe:** `note` is in `required` and typed `anyOf [string, null]`. The default value disappears: the model must always send the key, possibly as `null`. That is the "required + nullable" pattern from the README.

> **Try it:** add a nested `address: Address` model to `RefundRequest` and confirm the helper sets `additionalProperties: false` on the nested object too.

## 2.4 `tool_choice` and `parallel_tool_calls`

`auto` (default), `required` (≈ Claude's `any`), a specific function, `allowed_tools` (a per-turn subset, cache-friendly) or `none`. `parallel_tool_calls: false` limits each turn to zero or one call. [Theory](README.md#24-tool_choice-and-parallel_tool_calls)

In [ ]:
READ_ONLY = [{"type": "function", "name": "get_customer"}, {"type": "function", "name": "lookup_order"}]

tool_choice_examples = {
    "model decides (default)":         "auto",
    "must call at least one tool":     "required",
    "must call lookup_order":          {"type": "function", "name": "lookup_order"},
    "read-only subset, must call one": {"type": "allowed_tools", "mode": "required", "tools": READ_ONLY},
    "no tools this turn":              "none",
}
for why, choice in tool_choice_examples.items():
    print(f"{why:33} -> {json.dumps(choice)}")

# Chat Completions nests the forced function name one level deeper
print("\nChat Completions forced function ->", json.dumps({"type": "function", "function": {"name": "lookup_order"}}))

In [ ]:
# Parallel calls: several function_call items in ONE response (offline fake)
parallel = fake_response(
    fc("get_customer", {"email": "jane@example.com", "customer_id": None}, "call_1"),
    fc("lookup_order", {"order_id": "A-1001"}, "call_2"),
)
print("calls in one turn:", [(i.name, i.call_id) for i in parallel.output if i.type == "function_call"])
print("With parallel_tool_calls=False the model emits at most one of these per turn.")

In [ ]:
# (live) forcing a function even when it makes no sense
resp = client.responses.create(
    model=MODEL,
    tools=SUPPORT_TOOLS,
    tool_choice={"type": "function", "name": "lookup_order"},
    parallel_tool_calls=False,
    input="Hi there! Just saying hello.",
    **LIVE_KW,
)
print([(i.type, getattr(i, "name", None), getattr(i, "arguments", None)) for i in resp.output])

> **Observe:** a forced function is always called, even for "Hi there!", so the model has to make up an `order_id`. Forcing guarantees a call, not a sensible one. Use it for pipeline steps where the input always fits the tool.

> **Try it:** switch to `"required"`, then to the `allowed_tools` choice above with `mode: "auto"`, and compare which item types come back.

## 2.5 The client-side loop, and errors in tool outputs

No `stop_reason` here: loop while a completed response contains `function_call` items, replay every output item, return exactly one `function_call_output` per `call_id`, and turn every failure into an output string the model can act on (there is no `is_error` flag). [Theory](README.md#25-the-client-side-loop-and-errors-in-tool-outputs)

In [ ]:
ERROR_CODES = [  # (exception types, error code, retryable?) — first match wins
    (TimeoutError, "timeout", True),
    (PermissionError, "not_allowed", False),
    (LookupError, "not_found", False),
    ((ValueError, TypeError), "invalid_request", False),
]

def execute_tool(call, handlers: dict) -> str:
    """Run one function call and ALWAYS return a string for function_call_output. Never raises."""
    handler = handlers.get(call.name)
    if handler is None:
        return json.dumps({"ok": False, "error": "unknown_tool", "retryable": False,
                           "message": f"No tool named {call.name!r}. Available: {sorted(handlers)}"})
    try:
        args = json.loads(call.arguments)
    except json.JSONDecodeError as exc:                # only possible without strict mode
        return json.dumps({"ok": False, "error": "invalid_json", "retryable": False, "message": str(exc)})
    INTERNAL = {"ok": False, "error": "internal_error", "retryable": False,
                "message": "Unexpected failure. Apologize and offer a human handoff."}
    try:
        result = handler(**args)
    except Exception as exc:
        for exc_types, code, retryable in ERROR_CODES:
            if isinstance(exc, exc_types):
                return json.dumps({"ok": False, "error": code, "retryable": retryable, "message": str(exc)})
        return json.dumps(INTERNAL)            # unknown failure: don't leak internals (stack traces, SQL)
    try:
        return json.dumps({"ok": True, "result": result})
    except (TypeError, ValueError):            # result isn't JSON-serializable: our bug, not the model's
        return json.dumps(INTERNAL)


def lookup_order(order_id: str) -> dict:
    order = ORDERS.get(order_id)
    if order is None:
        raise LookupError(f"Order {order_id} not found. Order IDs look like A-1001; ask the user to check it.")
    return order

def flaky_inventory(sku: str) -> dict:
    raise TimeoutError("Inventory service timed out after 3 s.")

def tags_for(sku: str) -> set:
    return {"usb-c", "cable"}      # a bug: sets aren't JSON-serializable

demo_handlers = {"lookup_order": lookup_order, "inventory": flaky_inventory, "tags": tags_for}
for c in [fc("lookup_order", {"order_id": "A-1001"}, "c1"), fc("lookup_order", {"order_id": "A-9999"}, "c2"),
          fc("inventory", {"sku": "KB-1"}, "c3"), fc("delete_everything", {}, "c4"), fc("tags", {"sku": "KB-1"}, "c5")]:
    print(execute_tool(fake_response(c).output[0], demo_handlers))

In [ ]:
def find_refusal(response) -> str | None:
    """Return the refusal text if the response's message is a refusal, else None."""
    for item in response.output:
        if item.type == "message":
            for part in item.content:
                if part.type == "refusal":
                    return part.refusal
    return None


def run_agent(client, user_text: str, tools: list[dict], handlers: dict,
              instructions: str = "Use the tools for facts. Be brief.", max_turns: int = 6) -> str:
    """The bare Responses tool loop. Works with OpenAI() and with ScriptedClient."""
    input_list: list = [{"role": "user", "content": user_text}]
    for _ in range(max_turns):                                     # our own bound: max_tool_calls ignores functions
        response = client.responses.create(model=MODEL, instructions=instructions, tools=tools,
                                           input=input_list, **LIVE_KW)
        if response.status != "completed":
            raise RuntimeError(f"Response {response.status}: {response.incomplete_details}")
        refusal_text = find_refusal(response)                      # 1. refusals are "completed" too
        if refusal_text is not None:
            return f"[refused] {refusal_text}"                     #    surface it; never act on that turn
        input_list += response.output                              # 2. replay everything
        calls = [i for i in response.output if i.type == "function_call"]
        if not calls:                                              # 3. no calls -> done
            return response.output_text
        for call in calls:                                         # 4. one output per call_id
            input_list.append({"type": "function_call_output", "call_id": call.call_id,
                               "output": execute_tool(call, handlers)})
    raise RuntimeError("Agent exceeded max_turns")


scripted = ScriptedClient([
    fake_response(reasoning(), fc("lookup_order", {"order_id": "A-1001"}, "call_1"),
                  fc("lookup_order", {"order_id": "A-1002"}, "call_2")),            # two calls in parallel
    fake_response(msg("A-1001 was delivered; A-1002 has shipped and is on its way.")),
])
answer = run_agent(scripted, "Status of A-1001 and A-1002?", [lookup_order_tool], {"lookup_order": lookup_order})
print("answer:", answer)
print("requests sent:", len(scripted.requests))
print("2nd request input types:", [item_type(x) or "message(user)" for x in scripted.requests[1]["input"]])

# A refusal has no function calls and output_text == "": without the check above it would vanish silently
refused = ScriptedClient([fake_response(refusal("I'm sorry, I can't help with that."))])
print("refusal:", run_agent(refused, "Help me get into someone else's account.", [lookup_order_tool], {}))

> **Observe:** both outputs went back in the **same** next request, each with its own `call_id`, after the replayed reasoning and call items.

> **Try it:** script a third response that calls `lookup_order` again forever (repeat it 10 times in the list) and confirm the loop stops with `RuntimeError` after `max_turns`.

## 2.6 Structured Outputs: `text.format` and `responses.parse()`

Use strict **function calling** to connect the model to your systems, and **`text.format`** (`json_schema`) to shape its final answer. `client.responses.parse(text_format=Model)` sends the strict schema for you and returns `output_parsed`. [Theory](README.md#26-structured-outputs-textformat-and-responsesparse)

In [ ]:
class TicketClassification(BaseModel):
    category: Literal["billing", "shipping", "account", "unclear", "other"]
    category_detail: Optional[str]      # explain 'other' / 'unclear', else null
    order_id: Optional[str]             # null if the ticket has none

# The raw text.format a parse() call is equivalent to (schema generated from the model)
ticket_format = {
    "type": "json_schema",
    "name": "ticket_classification",
    "strict": True,
    "schema": openai.pydantic_function_tool(TicketClassification)["function"]["parameters"],
}
print(json.dumps({"text": {"format": ticket_format}}, indent=1)[:700], "...")
print("\nChat Completions equivalent key: response_format={'type': 'json_schema', 'json_schema': {...}}")

In [ ]:
# Offline: a structured answer arrives as JSON text in output_text; validate it into the model
structured = fake_response(msg('{"category":"billing","category_detail":null,"order_id":"A-1001"}'))
ticket = TicketClassification.model_validate_json(structured.output_text)
print(repr(ticket))

In [ ]:
# (live) the SDK helper does schema + parsing in one call
resp = client.responses.parse(
    model=MODEL,
    instructions="Classify the support ticket. Use 'unclear' if you cannot tell; use null for missing fields.",
    input="I was charged twice for order A-1001, please fix this.",
    text_format=TicketClassification,
    **LIVE_KW,
)
print(resp.status, "|", repr(resp.output_parsed))

> **Observe:** `output_parsed` is already a `TicketClassification`. No `json.loads`, no hand-written schema.

> **Try it:** send a ticket that has nothing to do with support ("What's the capital of France?") and check whether you get `unclear`. The docs warn that unrelated input can produce hallucinated fields when the schema gives the model no way out.

## 2.7 Refusals and incomplete responses

A refusal arrives as a `refusal` content part with `status == "completed"`; a cut-off answer has `status == "incomplete"` and `incomplete_details.reason`. Check both before parsing. [Theory](README.md#27-refusals-and-incomplete-responses)

In [ ]:
def read_structured(response) -> dict:
    """Return {'kind': 'ok'|'refusal'|'incomplete'|'not_completed', ...} without ever parsing a bad payload."""
    if response.status == "incomplete":
        return {"kind": "incomplete", "reason": response.incomplete_details.reason}
    if response.status != "completed":                       # failed, cancelled, queued, in_progress
        return {"kind": "not_completed", "status": response.status}
    for item in response.output:
        if item.type != "message" or item.phase not in (None, "final_answer"):
            continue                                         # skip reasoning, tool calls, commentary preambles
        for part in item.content:
            if part.type == "refusal":
                return {"kind": "refusal", "reason": part.refusal}
            if part.type == "output_text":
                return {"kind": "ok", "data": json.loads(part.text)}
    raise RuntimeError("No final-answer message in output")

TICKET_JSON = '{"category":"shipping","category_detail":null,"order_id":null}'
cases = {
    "ok":         fake_response(msg(TICKET_JSON)),
    "preamble":   fake_response(msg("Let me classify that.", phase="commentary"), msg(TICKET_JSON, phase="final_answer")),
    "refusal":    fake_response(refusal("I'm sorry, I can't help with that request.")),
    "truncated":  fake_response(msg('{"category":"ship'), status="incomplete", incomplete_reason="max_output_tokens"),
    "failed":     fake_response(msg(TICKET_JSON), status="failed"),
}
for name, r in cases.items():
    print(f"{name:10} status={r.status:10} ->", read_structured(r))

> **Observe:** the refusal case has `status == "completed"`, so a status check alone would have sent `"I'm sorry..."` into `json.loads`. With `responses.parse()`, the same case gives `output_parsed is None`. In the `preamble` case, `output_text` would be `"Let me classify that.{...}"` (it joins every message), so the reader takes only the `final_answer` message. A `failed` response is never parsed, even if it carries text.

> **Gotcha:** `responses.parse()` validates the text inside the call. A truncated payload (the `truncated` case) makes `parse()` itself raise `pydantic.ValidationError` before you can check `status`, so wrap it in `try/except ValidationError`, or use `create()` + `read_structured()` as the next section does.

## 2.8 Syntax vs semantic validation

Strict schemas remove **syntax** errors. **Semantic** errors (totals that don't add up, refunds above the order total, someone else's order) need your code: validators, cross-checks, retry with the specific error, human review. [Theory](README.md#28-syntax-vs-semantic-validation)

In [ ]:
class LineItem(BaseModel):
    description: str
    amount_usd: float

class InvoiceExtraction(BaseModel):
    invoice_number: Optional[str]        # null if the document has no invoice number
    line_items: list[LineItem]
    stated_total_usd: Optional[float]    # the total exactly as printed; null if no total is printed

INVOICE_FORMAT = {"type": "json_schema", "name": "invoice", "strict": True,
                  "schema": openai.pydantic_function_tool(InvoiceExtraction)["function"]["parameters"]}

def cents(usd: float) -> int:
    return round(usd * 100)              # money as integer cents: 100.01 - 100.00 is 0.010000000000005 in floats

def semantic_errors(inv: InvoiceExtraction) -> list[str]:
    computed = sum(cents(i.amount_usd) for i in inv.line_items)
    if inv.stated_total_usd is not None and abs(computed - cents(inv.stated_total_usd)) > 1:   # > 1 cent
        return [f"sum(line_items)={computed / 100:.2f} but stated_total_usd={inv.stated_total_usd}; "
                "re-check each amount."]
    return []

def extract_with_retry(client, document: str, max_retries: int = 2) -> InvoiceExtraction:
    input_list: list = [{"role": "user", "content": f"Extract the invoice data:\n\n{document}"}]
    errors: list[str] = []
    for _ in range(max_retries + 1):
        response = client.responses.create(model=MODEL, input=input_list, text={"format": INVOICE_FORMAT}, **LIVE_KW)
        result = read_structured(response)                       # 2.7: refusal / truncation first
        if result["kind"] != "ok":
            raise ValueError(f"No valid extraction ({result}); route to human review")
        invoice = InvoiceExtraction.model_validate(result["data"])  # syntax: guaranteed by the schema
        if invoice.stated_total_usd is None:                     # not in the source: a retry can't fix it
            raise ValueError("No total printed on the invoice; route to human review")
        errors = semantic_errors(invoice)                        # semantics: our job
        if not errors:
            return invoice
        input_list += response.output
        input_list.append({"role": "user", "content": "Validation failed:\n- " + "\n- ".join(errors)
                           + "\nCorrect only the affected fields. Don't invent values: use null for "
                             "invoice_number or stated_total_usd if they are not in the document."})
    raise ValueError(f"Still inconsistent after retries; route to human review: {errors}")

INVOICE_TEXT = "Invoice INV-77\nKeyboard ........ 120.00\nMouse ........... 29.90\nTOTAL USD ....... 149.90"
scripted = ScriptedClient([
    fake_response(msg('{"invoice_number":"INV-77","line_items":[{"description":"Keyboard","amount_usd":120.0},'
                      '{"description":"Mouse","amount_usd":2.99}],"stated_total_usd":149.9}')),   # misread 29.90
    fake_response(msg('{"invoice_number":"INV-77","line_items":[{"description":"Keyboard","amount_usd":120.0},'
                      '{"description":"Mouse","amount_usd":29.9}],"stated_total_usd":149.9}')),
])
invoice = extract_with_retry(scripted, INVOICE_TEXT)
print("final:", invoice)
print("attempts:", len(scripted.requests))
print("feedback sent:", scripted.requests[1]["input"][-1]["content"].splitlines()[1])

In [ ]:
# (live) the same pipeline against the API
print(extract_with_retry(client, INVOICE_TEXT))

> **Observe:** the first scripted answer was perfectly schema-valid and still wrong. Only the code-level check caught it, and the retry carried the *specific* discrepancy.

> **Try it:** change `INVOICE_TEXT` so the printed total really is wrong (e.g. `TOTAL USD ... 159.90`). Retries can't fix a source document; decide whether your pipeline should flag it instead.

## Mini-project: customer-support agent loop

Combine everything: strict tools (2.2–2.3), phase-based `allowed_tools` (2.4), the loop with error outputs (2.5) and code-level business rules for the irreversible `process_refund` (2.8).

Rules enforced **in code**, not in the prompt:

1. Before a customer is verified, only `get_customer` and `lookup_order` are callable (`allowed_tools`).
2. Order details are only returned for the verified customer's own orders (unverified callers get the status only).
3. A refund needs a verified customer, an order that belongs to that customer, `status == "delivered"`, and `0 < amount ≤ remaining`, where *remaining* is the order total minus refunds already issued (so repeating a refund can't exceed the total).
4. Refunds above 100 USD are never executed by the agent; they return `requires_human_approval`.

(Matching an email is not real identity verification. In production, verification comes from your auth layer, not from what the user types.)

In [ ]:
class SupportSession:
    """Per-conversation state that the CODE trusts (the model can't write to it directly)."""
    def __init__(self) -> None:
        self.verified_customer_id: str | None = None
        self.refunds: list[dict] = []

AUTO_REFUND_LIMIT_USD = 100.0

def make_handlers(session: SupportSession) -> dict:
    def get_customer(email: str | None, customer_id: str | None) -> dict:
        if (email is None) == (customer_id is None):
            raise ValueError("Pass exactly one of email or customer_id and set the other to null.")
        if customer_id is not None:
            customer = CUSTOMERS.get(customer_id)
        else:
            customer = next((c for c in CUSTOMERS.values() if c["email"].lower() == email.lower()), None)
        if customer is None:
            raise LookupError("No matching customer. Ask the user to double-check their email or customer ID.")
        session.verified_customer_id = customer["customer_id"]
        return customer

    def scoped_lookup_order(order_id: str) -> dict:
        order = lookup_order(order_id)                       # raises LookupError if unknown
        if session.verified_customer_id is None:
            return {"order_id": order_id, "status": order["status"]}   # unverified: status only
        if order["customer_id"] != session.verified_customer_id:
            raise PermissionError("This order is not on the verified customer's account. Do not share or refund it.")
        return order

    def process_refund(order_id: str, amount_usd: float, reason: str) -> dict:
        if session.verified_customer_id is None:
            raise PermissionError("Customer not verified. Call get_customer first.")
        order = scoped_lookup_order(order_id)                # ownership check (raises PermissionError)
        if order["status"] != "delivered":
            raise ValueError(f"Order status is '{order['status']}'; only delivered orders can be refunded.")
        already = sum(r["amount_usd"] for r in session.refunds if r["order_id"] == order_id)
        remaining = round(order["total_usd"] - already, 2)
        if not 0 < amount_usd <= remaining:
            raise ValueError(f"amount_usd must be > 0 and <= the remaining refundable amount ({remaining:.2f} "
                             f"of {order['total_usd']:.2f}; {already:.2f} already refunded).")
        if amount_usd > AUTO_REFUND_LIMIT_USD:
            return {"status": "requires_human_approval", "order_id": order_id, "amount_usd": amount_usd}
        refund = {"status": "refunded", "refund_id": f"R-{len(session.refunds) + 1:04d}",
                  "order_id": order_id, "amount_usd": amount_usd, "reason": reason}
        session.refunds.append(refund)
        return refund

    return {"get_customer": get_customer, "lookup_order": scoped_lookup_order, "process_refund": process_refund}


SUPPORT_INSTRUCTIONS = (
    "You are a customer-support agent for an online electronics shop. Verify the customer with "
    "get_customer before discussing their account or refunding anything. Use tools for facts and never "
    "invent order data. When a tool returns ok=false, read 'error' and 'message', then fix the call, ask "
    "the user, or explain the limitation. Keep replies under 80 words."
)

def run_support_agent(client, user_text: str, session: SupportSession | None = None,
                      max_turns: int = 6, verbose: bool = True) -> str:
    session = session or SupportSession()
    handlers = make_handlers(session)
    input_list: list = [{"role": "user", "content": user_text}]
    for turn in range(1, max_turns + 1):
        # Phase gating with a STABLE tools list (cache-friendly): read-only until verified
        tool_choice = ("auto" if session.verified_customer_id
                       else {"type": "allowed_tools", "mode": "auto", "tools": READ_ONLY})
        response = client.responses.create(model=MODEL, instructions=SUPPORT_INSTRUCTIONS, tools=SUPPORT_TOOLS,
                                           tool_choice=tool_choice, input=input_list, **LIVE_KW)
        if response.status != "completed":
            raise RuntimeError(f"Turn {turn}: response {response.status} ({response.incomplete_details})")
        reason = find_refusal(response)
        if reason:                                             # never act on a refused turn
            return f"[refused] {reason}"
        input_list += response.output
        calls = [i for i in response.output if i.type == "function_call"]
        if not calls:
            return response.output_text
        for call in calls:
            output = execute_tool(call, handlers)
            if verbose:
                print(f"turn {turn}: {call.name}({call.arguments}) -> {output[:110]}")
            input_list.append({"type": "function_call_output", "call_id": call.call_id, "output": output})
    raise RuntimeError("Support agent exceeded max_turns; hand off to a human.")

In [ ]:
# Offline run 1: happy path (parallel lookups, then a refund under the limit)
session = SupportSession()
scripted = ScriptedClient([
    fake_response(reasoning(),
                  fc("get_customer", {"email": "jane@example.com", "customer_id": None}, "call_1"),
                  fc("lookup_order", {"order_id": "A-1001"}, "call_2")),
    fake_response(fc("process_refund", {"order_id": "A-1001", "amount_usd": 9.9, "reason": "damaged"}, "call_3")),
    fake_response(msg("Done: I refunded 9.90 USD for the broken USB-C cable (refund R-0001).")),
])
print(run_support_agent(scripted, "I'm jane@example.com. The cable in A-1001 arrived broken, refund 9.90 please.", session))
print("tool_choice per turn:", [r["tool_choice"] if isinstance(r["tool_choice"], str) else r["tool_choice"]["type"]
                                for r in scripted.requests])
print("refunds recorded:", session.refunds)

# Later refunds are checked against what is LEFT to refund (89.90 - 9.90 = 80.00), so repeats can't exceed
# the total. Recognizing an exact duplicate (a retried HTTP request) also needs an idempotency key per request.
replay = make_handlers(session)["process_refund"]
for amount in (80.0, 9.9):
    try:
        print(f"refund {amount}:", replay("A-1001", amount, "damaged")["status"])
    except ValueError as exc:
        print(f"refund {amount}: blocked ->", exc)

In [ ]:
# Offline run 2: the model tries to refund someone else's order -> blocked in code, error goes back as data
session = SupportSession()
scripted = ScriptedClient([
    fake_response(fc("get_customer", {"email": "jane@example.com", "customer_id": None}, "call_1"),
                  fc("lookup_order", {"order_id": "A-2001"}, "call_2")),
    fake_response(fc("process_refund", {"order_id": "A-2001", "amount_usd": 35.5, "reason": "other"}, "call_3")),
    fake_response(msg("I'm sorry, order A-2001 isn't on your account, so I can't refund it.")),
])
print(run_support_agent(scripted, "I'm jane@example.com, refund order A-2001 to me.", session))
print("refunds recorded:", session.refunds)

> **Observe:** in run 2 every call satisfied the strict schemas (a valid `order_id`, a number, an allowed `reason`), yet the lookup was refused and the refund was blocked, and the model received both refusals as data. That's the syntax/semantic split from 2.8 applied to tool calls, and it holds regardless of what the prompt says. (A real model would usually stop after the failed lookup; the script pushes on to show the second guard.)

Now the real thing **(live)**:

In [ ]:
# (live) the same agent against the API
live_session = SupportSession()
print(run_support_agent(client, "Hi, I'm jane@example.com. The USB-C cable in order A-1001 arrived broken; "
                                "please refund 9.90 USD.", live_session))
print("refunds:", live_session.refunds)

> **Try it:**
> - Ask for a 150 USD refund on `A-1002` (shipped, not delivered) and then on a delivered order above the limit. Which rule fires?
> - Set `parallel_tool_calls=False` in `run_support_agent` and count the turns.
> - Remove the `allowed_tools` gating and ask for a refund without giving an email. What stops the refund now?

## Recap

- A call is a `function_call` item; the result is a `function_call_output` item with the same **`call_id`**. `arguments` is a JSON string.
- Responses tools are **flat**; Chat Completions tools (and `openai.pydantic_function_tool()`) are **nested**.
- Strict mode: `additionalProperties: false` everywhere, **every** property required, optional = nullable. Set `strict: true` explicitly.
- `tool_choice`: `auto` / `required` / a function / `allowed_tools` / `none`; `parallel_tool_calls: false` = at most one call per turn.
- The loop replays every output item, answers every `call_id` (errors included, as data), and has its own `max_turns`.
- `text.format` / `responses.parse(text_format=...)` shape the final answer; check `status` and refusals before parsing.
- Schemas guarantee shape, not truth: business rules live in your code.

Next: [homework](02_homework.ipynb)